<a href="https://colab.research.google.com/github/toptee2020/toptee2020/blob/main/Week_8_Lecture_Master_Locked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8: Pandas and Intro to Data Wrangling

Here's the journey so far:

| Week | You learned how to... |
|---|---|
| 1-2 | **store** data (values, lists, dictionaries) |
| 4-5 | **decide** and **repeat** (`if`, loops) |
| 6 | **package** code into functions |
| 7 | **borrow** code (libraries) and do math on whole arrays (NumPy) |
| **8** | **work with real tables of data** (pandas) |

This is the week Python becomes a data science tool. **pandas** is the library data scientists use to work with tables: spreadsheets, survey results, sales records, sensor readings, anything with rows and columns. Today you'll load a real dataset of thousands of rows and start asking it questions.

The good news: almost every piece of pandas is built from something you already know. Dictionaries from Week 2. Functions from Week 6. NumPy from last week. We'll point out each connection as we go.

---
## How to use this notebook

Work through it **on your own, at your own pace**. Everything is explained in writing, so take your time. It should take about **an hour**, plus the Practice Zone.

| Section | Rough time |
|---|---|
| Block 1 · Tables, rows, and columns | 5 min |
| Block 2 · Build a tiny table yourself | 15 min |
| Block 3 · Load a real dataset | 10 min |
| Block 4 · Get to know your data | 10 min |
| Block 5 · Grab a column | 10 min |
| Block 6 · NumPy tricks work on columns | 8 min |
| Block 7 · pandas errors | 7 min |
| Block 8 · Sneak peek: next week | 3 min |
| Practice Zone (ungraded) | as long as you like |

**Ground rules (same as always):**

1. **Run every code cell, one at a time, top to bottom** (`Shift + Enter`).
2. **Guess before you run.** The 🟧 GUT CHECKs ask you to predict first.
3. **Cells marked 🟥 fail on purpose.** Read the **last line** of the error, then keep going.

## Quick Reference

| Term | What it means | Example |
|---|---|---|
| **pandas** | the library for working with tables of data | `import pandas as pd` |
| **DataFrame** | a whole table: rows and columns | `housing` |
| **column** | one variable, measured for every row | `housing["population"]` |
| **row** | one thing being measured | one neighborhood |
| **Series** | a single column on its own | `housing["population"]` |
| **CSV file** | a plain-text spreadsheet, values separated by commas | `sales.csv` |
| **`pd.read_csv()`** | loads a CSV file into a DataFrame | `pd.read_csv("file.csv")` |
| **`.head()`** | shows the first few rows | `housing.head()` |
| **`.shape`** | how many rows and columns | `housing.shape` |
| **`.describe()`** | a summary of every number column | `housing.describe()` |

---
# Block 1 · Tables, Rows, and Columns

You've seen tables your whole life: a spreadsheet, a sports standings page, a gradebook. Data scientists think about tables in a very specific way:

- Each **row** is **one thing** being measured: one person, one sale, one day, one neighborhood.
- Each **column** is **one fact** about every one of those things: age, price, temperature, population.

Here's a tiny example, where each row is one coffee shop:

| shop | city | price | rating |
|---|---|---|---|
| Bean There | Eugene | 4.50 | 4.6 |
| Grounds Up | Portland | 5.25 | 4.2 |
| Daily Grind | Salem | 3.95 | 4.8 |

> ⭐ **KEY IDEA:** Before you analyze any dataset, ask: **"What is one row?"** In the table above, one row is one coffee shop. That sounds obvious, but in real datasets it's often surprisingly tricky, and getting it wrong leads to wrong answers. (Is one row a customer, or one purchase? A city, or one day in one city?)

In pandas, a whole table is called a **DataFrame**. Let's build one.

---
# Block 2 · Build a Tiny Table Yourself

First, import pandas with its standard nickname. Remember the data science "uniform" from last week:

In [ ]:
# Block 2 · Cell 1
import pandas as pd      # pandas, with the nickname everyone uses
import numpy as np       # we'll use NumPy a little too

### Quick review: dictionaries

pandas is built on an idea you learned back in Week 2: the **dictionary**. Since it's been a while, here's a refresher.

> 🟦 **DEFINITION:** A **dictionary** stores information as **key-value pairs**. The **key** is a label, and the **value** is the information stored under that label. Think of a real dictionary: you look up a word (the key) to find its definition (the value).

A dictionary is written with **curly braces** `{ }`. Inside, each pair is written as `key: value`, and pairs are separated by commas. Keys are usually strings in quotes.

```python
shop = {"name": "Bean There", "city": "Eugene", "price": 4.50}
#        key      value        key     value     key     value
```

To **look up** a value, put its key in **square brackets** after the dictionary's name: `shop["city"]` gives back `"Eugene"`. Notice the shapes: you *make* a dictionary with curly braces, but you *look things up* with square brackets.

In [ ]:
# Block 2 · Cell 2
shop = {"name": "Bean There", "city": "Eugene", "price": 4.50}   # one coffee shop, as key-value pairs

print(shop["name"])     # look up the value stored under the key "name"
print(shop["price"])    # look up the price
print(shop.keys())      # all the keys (the labels)

Two rules from Week 2 matter a lot this week:

1. **Keys must match exactly.** `shop["Name"]` fails with a `KeyError`, because the key is `"name"` with a lowercase n. Capitalization, spelling, and spaces all count.
2. **A value can be anything**, including a whole **list**. That's the step that turns a dictionary into a table.

Here's the leap. The dictionary above describes **one** shop. What if each key held a **list** of values, one for every shop?

In [ ]:
# Block 2 · Cell 3
shops = {
    "name":  ["Bean There", "Grounds Up", "Daily Grind"],   # a list of three names
    "price": [4.50, 5.25, 3.95],                            # a list of three prices, in the same order
}

print(shops["price"])       # looking up a key now gives back a whole list
print(shops["price"][1])    # the price at position 1: the second shop, Grounds Up

> 🟧 **GUT CHECK:** Why does `shops["price"][1]` give `5.25` and not `4.50`? (Remember: list positions start at 0, so position 1 is the **second** item.)

Look at what that dictionary really is: each **key** is a label, like a column header, and each **list** is a column of values, lined up so that position 0 in every list describes the same shop. That's a table! It's just written sideways. pandas turns it right side up.

### A table is a dictionary of columns

Here's the trick: **a DataFrame is built from a dictionary where each key is a column name and each value is a list of that column's data.**

In [ ]:
# Block 2 · Cell 4
coffee_data = {
    "shop":   ["Bean There", "Grounds Up", "Daily Grind"],   # column 1
    "city":   ["Eugene", "Portland", "Salem"],               # column 2
    "price":  [4.50, 5.25, 3.95],                            # column 3
    "rating": [4.6, 4.2, 4.8],                               # column 4
}

coffee = pd.DataFrame(coffee_data)   # turn the dictionary into a table
coffee                               # Colab displays the last line as a nice table

Look at what pandas did:

- Each **dictionary key** became a **column name** across the top.
- Each **list** became a **column** of values.
- Each **position** in the lists became a **row**. Item 0 of every list is row 0, item 1 is row 1, and so on.

And on the far left, pandas added row labels: `0`, `1`, `2`. These are called the **index**. For now, just think of them as row numbers (starting from 0, like list positions).

> 🟥 **ERROR TO WATCH FOR:** Every list must be the **same length**, because every row needs a value in every column. This cell fails on purpose:

In [ ]:
# Block 2 · Cell 5
# 🟥 Fails on purpose: 3 shops but only 2 prices
broken = pd.DataFrame({
    "shop": ["Bean There", "Grounds Up", "Daily Grind"],
    "price": [4.50, 5.25],
})

The last line says `ValueError: All arrays must be of the same length`. Same idea as last week, when NumPy couldn't add a 3-item array to a 2-item array.

> 🟩 **TIP:** In real work, you'll rarely type data in by hand like this. Usually you **load** it from a file, which is the next block. But building a tiny table yourself is the best way to see what a DataFrame really is: a bunch of named columns, lined up side by side.

---
# Block 3 · Load a Real Dataset

Most data arrives as a **CSV file** ("comma-separated values"). It's just a plain-text spreadsheet: one line per row, with commas between the values. Every spreadsheet program (Excel, Google Sheets) can save as CSV, so it's the most common data format in the world.

Colab comes with a few sample datasets already loaded. We'll use one about **housing in California**.

> 🟪 **DATASET:** This data comes from the **1990 U.S. Census**. Each row is one **block group**: a small, neighborhood-sized area in California. For each neighborhood, it records things like how old the houses are, how many people live there, the median income, and the median house value.

In [ ]:
# Block 3 · Cell 1
housing = pd.read_csv("sample_data/california_housing_train.csv")   # load the file into a DataFrame

Nothing printed? That's expected. Just like `total = sum(temps)`, we **saved** the result in a variable called `housing` instead of displaying it. `pd.read_csv()` is a function from the pandas library (dot notation, from last week!) that reads a CSV file and gives back a DataFrame.

### Look at the first few rows

A real dataset is too big to print all at once. `.head()` shows just the first 5 rows:

In [ ]:
# Block 3 · Cell 2
housing.head()     # the first 5 rows

You can ask for a different number of rows by putting it in the parentheses, and `.tail()` shows the **last** rows instead:

In [ ]:
# Block 3 · Cell 3
housing.head(3)    # just the first 3 rows

In [ ]:
# Block 3 · Cell 4
housing.tail()     # the last 5 rows

### How big is it?

In [ ]:
# Block 3 · Cell 5
print(housing.shape)     # (number of rows, number of columns)

`.shape` gives back two numbers in parentheses: **(rows, columns)**. That's thousands of neighborhoods and 9 columns. Imagine scrolling through that in a spreadsheet!

> 🟥 **ERROR TO WATCH FOR:** Notice that `.shape` has **no parentheses**, but `.head()` does. Why?
>
> - `.head()` is a **method**: an action you ask the table to *do* ("show me your first rows"). Actions need `()`.
> - `.shape` is a **fact** the table already *knows* about itself, just stored and waiting. No action needed, so no `()`.
>
> Writing `housing.shape()` gives a confusing `TypeError`. You'll see it in Block 7.

### What are the columns called?

In [ ]:
# Block 3 · Cell 6
print(housing.columns)    # the name of every column (also a fact: no parentheses)

Here's what each column means:

| Column | What it measures (for each neighborhood) |
|---|---|
| `longitude`, `latitude` | where it is on a map |
| `housing_median_age` | the typical age of the houses, in years |
| `total_rooms` | total rooms in all the houses combined |
| `total_bedrooms` | total bedrooms in all the houses combined |
| `population` | how many people live there |
| `households` | how many households live there |
| `median_income` | typical household income, **in tens of thousands of dollars** (so `3.5` means about $35,000) |
| `median_house_value` | typical house value, in dollars |

> 🟧 **GUT CHECK:** `median_income` is measured in tens of thousands of dollars. That's the kind of detail that's easy to miss, and it would make your answers off by a factor of 10,000! Always find out what the **units** are before you trust a number. (Real datasets usually come with a description like this table, often called a **data dictionary**.)

---
# Block 4 · Get to Know Your Data

Before you answer any questions with a dataset, take a few minutes to **get to know it**. Data scientists do this every single time. Two methods make it easy.

### `.info()`: what's in each column?

In [ ]:
# Block 4 · Cell 1
housing.info()

For each column, `.info()` tells you:

- **Non-Null Count:** how many rows actually have a value. If a column has fewer than the total number of rows, some values are **missing**. (Missing data is extremely common in real life. We'll learn to deal with it later.)
- **Dtype:** the type of data in the column. `float64` means decimal numbers, `int64` means whole numbers, and `object` usually means text. (These are the NumPy types from last week, under the hood.)

### `.describe()`: a summary of every number column

In [ ]:
# Block 4 · Cell 2
housing.describe()

In one line, `.describe()` calculates a summary for **every** number column at once:

| Row | Means |
|---|---|
| `count` | how many values |
| `mean` | the average |
| `std` | how spread out the values are (we'll cover this later) |
| `min` | the smallest value |
| `25%`, `50%`, `75%` | the values a quarter, half, and three quarters of the way up, when sorted. `50%` is the **median**. |
| `max` | the largest value |

> 🟩 **TIP:** The numbers may show up in scientific notation, like `1.7e+04`. That means 1.7 × 10⁴, or 17,000. It's just a compact way to write big numbers.

### Getting curious

`.describe()` is a great place to spot things that make you say **"I wonder..."** Look at the `max` row for `median_house_value` and `housing_median_age`.

> 🟧 **GUT CHECK:** The highest house value is **500,001**. Not 500,000, not 2,000,000: exactly 500,001. And the oldest typical house age is exactly **52**. Does that seem like a coincidence?
>
> It isn't. Whoever prepared this dataset **capped** those columns: any neighborhood worth more than $500,000 was just recorded as 500,001. So the data can't tell you which neighborhoods were *really* expensive. This is exactly the kind of thing you'd never notice if you skipped straight to your analysis. Always look before you leap!

---
# Block 5 · Grab a Column

Most questions are about **one column** at a time: "What's the average house value?" "What's the biggest population?" To grab a column, use square brackets with the column's name, **exactly like getting a value from a dictionary**:

In [ ]:
# Block 5 · Cell 1
house_values = housing["median_house_value"]   # grab one column by its name
house_values.head()                            # look at the first few values

> 🟦 **DEFINITION:** A single column on its own is called a **Series**. Think of a Series as **a NumPy array with labels**: the values are the array, and the row numbers on the left are the labels.

### Summarize a column

Because a Series is basically a NumPy array, all of last week's summary methods work:

In [ ]:
# Block 5 · Cell 2
print(house_values.mean())     # average house value
print(house_values.median())   # middle house value (pandas has .median(), NumPy arrays didn't!)
print(house_values.max())      # highest
print(house_values.min())      # lowest

Those averages have a lot of decimal places. Use `round()` from Week 6:

In [ ]:
# Block 5 · Cell 3
avg_value = round(house_values.mean())       # round to a whole dollar
print("Average house value:", avg_value)

> 🟧 **GUT CHECK:** Is the average between the min and the max? It has to be. Is the mean bigger or smaller than the median? When a few very high values pull the average up, the mean ends up **above** the median. (Remember the incomes example from Week 6's practice?)

You don't have to save the column in its own variable first. This does the same thing in one line:

In [ ]:
# Block 5 · Cell 4
print(housing["population"].max())    # grab the column, then ask for its max

Read `housing["population"].max()` from left to right: **"from the housing table, grab the population column, then find its max."**

### Grabbing more than one column

To grab several columns, put a **list** of names inside the square brackets. That's why you see **two** sets of brackets: the outer ones mean "grab," and the inner ones make a list.

In [ ]:
# Block 5 · Cell 5
housing[["median_income", "median_house_value"]].head()   # a smaller table with just 2 columns

> 🟩 **TIP:** One set of brackets gives you **one column** (a Series). Two sets of brackets give you a **smaller table** (a DataFrame). `.describe()` works on both!

In [ ]:
# Block 5 · Cell 6
housing[["median_income", "median_house_value"]].describe()   # summary of just these 2 columns

---
# Block 6 · Last Week's NumPy Tricks Work on Columns

Here's the payoff for learning NumPy first. Everything you did with arrays works on columns.

### Math on every value at once

`median_income` is in tens of thousands of dollars. To convert every neighborhood's income into regular dollars, multiply the whole column by 10,000. No loop needed:

In [ ]:
# Block 6 · Cell 1
income_dollars = housing["median_income"] * 10000   # every value multiplied at once
income_dollars.head()

### Ask a True/False question of every row

In [ ]:
# Block 6 · Cell 2
expensive = housing["median_house_value"] > 400000   # is each neighborhood over $400,000?
expensive.head()

### Count the Trues

Just like last week: `True` counts as 1 and `False` as 0, so `.sum()` counts the Trues.

In [ ]:
# Block 6 · Cell 3
print((housing["median_house_value"] > 400000).sum())   # how many neighborhoods are over $400,000?

And you can turn that count into a **percent** by dividing by the total number of rows:

In [ ]:
# Block 6 · Cell 4
num_expensive = (housing["median_house_value"] > 400000).sum()   # count of expensive neighborhoods
num_rows = len(housing)                                         # total neighborhoods (len works on tables too!)
percent = round(num_expensive / num_rows * 100, 1)              # turn it into a percent
print(percent, "percent of neighborhoods are over $400,000")

> ⭐ **KEY IDEA:** Look at how much you just did: loaded thousands of rows, grabbed a column, asked a question of every single row, counted the answers, and turned it into a percent. Six weeks ago, that would have taken a loop, an `if`, an accumulator, and a lot of typing. This is why data scientists use pandas.

---
# Block 7 · pandas Errors

pandas error messages are **long**, often 20 or 30 lines, and most of it is pandas' own code. Don't panic. **Scroll to the last line.** It almost always tells you exactly what's wrong.

### 1. The column name doesn't match (the #1 pandas error)

In [ ]:
# Block 7 · Cell 1
# 🟥 Fails on purpose: wrong capitalization
housing["Population"].max()

The last line is `KeyError: 'Population'`. Just like with dictionaries, the key must match **exactly**: same spelling, same capitalization, same underscores, no extra spaces.

**Fix:** run `print(housing.columns)` and copy the name exactly. (Copying beats retyping!)

### 2. Parentheses on a fact

In [ ]:
# Block 7 · Cell 2
# 🟥 Fails on purpose: shape is a fact, not an action
housing.shape()

`TypeError: 'tuple' object is not callable`. That's a strange-sounding message! Here's what it means: `housing.shape` is **already** the answer, `(rows, columns)`. Adding `()` tells Python to *run* that answer like a function, and you can't run a pair of numbers. "Not callable" means "you can't put `()` after this."

**Fix:** remove the parentheses: `housing.shape`.

### 3. Forgetting the import (or forgetting to re-run it)

If Colab restarts or disconnects, it forgets everything. Then any line with `pd` fails with `NameError: name 'pd' is not defined`, and any line with `housing` fails with `NameError: name 'housing' is not defined`.

**Fix:** scroll up and re-run the cells with `import pandas as pd` (Block 2) and `pd.read_csv(...)` (Block 3), in that order.

> 🟩 **TIP:** In your own notebooks, put `import pandas as pd` **and** your `pd.read_csv(...)` line in the first two cells. Then after any restart, re-run those two cells and you're back in business.

---
# Block 8 · Sneak Peek: Next Week

In Block 6, you **counted** the expensive neighborhoods. Next week, you'll learn to **keep only those rows** so you can study them, using the same trick as NumPy filtering last week. Run this cell and see if you can guess what it does:

In [ ]:
# Block 8 · Cell 1
pricey = housing[housing["median_house_value"] > 400000]   # "housing, where house value is over 400,000"
pricey.head()

That's a whole new table containing only the expensive neighborhoods. Next week we'll dig into how it works, plus sorting and counting categories, before the midterm.

---
> ⭐ **KEY IDEAS FROM THIS WEEK**
> 1. pandas works with tables called **DataFrames**. Always ask: **what is one row?**
> 2. A DataFrame is like a **dictionary of columns**: `pd.DataFrame({"col": [...]})`.
> 3. `pd.read_csv("file.csv")` loads a dataset. `.head()`, `.shape`, and `.columns` show you what you've got.
> 4. `.info()` and `.describe()` help you get to know your data. Look for surprises, like capped values!
> 5. `housing["column"]` grabs one column (a **Series**). `.mean()`, `.median()`, `.max()`, `.min()` summarize it.
> 6. NumPy tricks work on columns: math on every value, True/False questions, and `.sum()` to count.
> 7. Long error? **Read the last line.** `KeyError` usually means a column name doesn't match exactly.

---
# Practice Zone (Ungraded)

Try each before peeking at the solution.

> 🟩 **TIP:** If you get `NameError: name 'housing' is not defined`, re-run the import cell in Block 2 and the `read_csv` cell in Block 3.

### Problem 1 · Build a table

Make a DataFrame called `pets` with two columns: `"name"` (`"Charlie"`, `"Luna"`, `"Max"`) and `"age"` (`5`, `2`, `9`). Display it.

In [ ]:
# Practice 1
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
pets = pd.DataFrame({
    "name": ["Charlie", "Luna", "Max"],
    "age": [5, 2, 9],
})
pets
```

</details>

### Problem 2 · Summarize your table

Using `pets`, print the average age and the oldest age.

In [ ]:
# Practice 2
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
print(pets["age"].mean())   # about 5.3
print(pets["age"].max())    # 9
```

</details>

### Problem 3 · Peek at the data

Show the first 7 rows of `housing`.

In [ ]:
# Practice 3
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
housing.head(7)
```

</details>

### Problem 4 · How big?

Print the number of rows and columns in `housing`. Then print **just** the number of rows using `len()`.

In [ ]:
# Practice 4
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
print(housing.shape)
print(len(housing))
```

</details>

### Problem 5 · One column

Grab the `households` column and print its median.

In [ ]:
# Practice 5
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
print(housing["households"].median())
```

</details>

### Problem 6 · Round it

Print the average `housing_median_age`, rounded to 1 decimal place.

In [ ]:
# Practice 6
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
print(round(housing["housing_median_age"].mean(), 1))
```

</details>

### Problem 7 · Two columns

Show the first 5 rows of just the `population` and `households` columns.

In [ ]:
# Practice 7
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
housing[["population", "households"]].head()
```

</details>

### Problem 8 · Count

How many neighborhoods have a `housing_median_age` of **50 or more**?

In [ ]:
# Practice 8
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
print((housing["housing_median_age"] >= 50).sum())
```

</details>

### Problem 9 · Debug it

Each line has a problem. Run it, read the last line of the error, and fix it.

```python
print(housing["median_House_value"].mean())
print(housing.columns())
print(housing["population"].max)
```

(Careful: the third one doesn't give an error. But is the output what you wanted?)

In [ ]:
# Practice 9
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
# KeyError: the column is all lowercase
print(housing["median_house_value"].mean())

# TypeError: columns is a fact, not an action, so no ()
print(housing.columns)

# No error, but it prints a description of the method instead of the answer!
# .max() is an action, so it NEEDS parentheses
print(housing["population"].max())
```

</details>

### Problem 10 · People per household

Each neighborhood has a `population` and a number of `households`. Divide one column by the other to get **people per household** for every neighborhood, all at once. Then print the average and the max.

Gut check: a typical household has 2 to 4 people. Is the average close to that? Is the max believable? What might explain a very high max? (Hint: think about dorms, prisons, or military bases.)

In [ ]:
# Practice 10
# Your plan here, as comments:

<details>
<summary>Show a solution</summary>

```python
people_per_household = housing["population"] / housing["households"]   # column divided by column, row by row
print(round(people_per_household.mean(), 2))
print(round(people_per_household.max(), 2))
# A very high max probably means a place where many people live in
# "group quarters" (like a dorm or barracks) counted as few households.
# Real data is full of surprises like this!
```

</details>